Universidad Torcuato Di Tella

*Licenciatura en Tecnología Digital*

**Métodos Computacionales — Reconstrucción y regularización en imágenes tomográficas**
# **Trabajo Práctico — Ejercicio N°7: La reconstrucción como proyección ortogonal**

Antes de comenzar a resolver, importamos los módulos que utilizaremos en el desarrollo.

In [1]:
# Importamos los módulos
import numpy as np
import matplotlib.pyplot as plt

# Pandas es para armar la tabla
import pandas as pd

Además, corremos los bloques necesarios de ejercicios anteriores para poder realizar este ejercicio:

In [2]:
# Ejercicio 3

# Definimos `X_fantoma`
X_fantoma:np.ndarray = np.array([
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
[0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
[0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

# Aplanamos la matriz `X_fantoma` para dar lugar a `x_fantoma`
x_fantoma = X_fantoma.reshape(-1)

In [3]:
# Ejercicio 3
def mascara_rayo(n:int, direccion:tuple[int,int], s:int):
    r'''
    Recibe:
    - `n` -> dimensión de la matriz (se asume cuadrada).
    - `direccion` -> dirección del rayo (fija una familia de rayos paralelos).
    - `s` -> distancia del rayo al origen (fija un rayo).

    Devuelve:
    - La matriz que representa la mascara del rayo sobre la imagen.
    '''
    res:np.ndarray = np.zeros(shape=(n,n), dtype=np.int8)
    for i in range(0,len(res)):
        for j in range(0,len(res)):
            if (direccion[0] * i) + (direccion[1] * j) == s:
                res[i][j] = 1
    return res

In [4]:
# Ejercicio 3
def matriz_proyeccion(n:int, direcciones:list[tuple[int, int]]):
    r'''
    
    Recibe:
    - `n` -> dimensión de la matriz (se asume cuadrada).
    - `direcciones` -> vector de direcciones a utilizar.

    Devuelve:
    - La matriz compuesta por las máscaras de todos los rayos correspondientes.
    - La lista de todos los rayos generados, con el ángulo y el s correspondiente.
    '''
    A:list[list[int]] = list()
    res:list[tuple[tuple[int,int], int]] = list()
    for dir in direcciones:
        rayos:set[int] = set()
        for i in range(n):
            for j in range(n):
                s = dir[0] * i + dir[1] * j
                rayos.add(s)
        rayos_lista:list[int] = list(rayos)
        rayos_lista.sort()
        rayos_array:np.ndarray = np.array(rayos_lista)

        for rayo in rayos_array:
            fila_A = mascara_rayo(n, dir, rayo)
            fila_A = fila_A.reshape(-1)
            A.append(fila_A) # type: ignore
            res.append((dir, rayo))

    return A, res

In [5]:
# Ejercicio 4

# Para definir dimensiones de la matriz (nxn)
n = 8

D2 = [(1,0),(0,1)]
D4 = [(1,0),(0,1),(1,1),(1,-1)]
D7 = [(1,0),(0,1),(1,1),(1,-1),(2,1),(1,2),(3,1)]

# Construímos A2, A4, A7
A2 = matriz_proyeccion(n, D2)[0]
A4 = matriz_proyeccion(n, D4)[0]
A7 = matriz_proyeccion(n, D7)[0]

In [6]:
# Ejercicio 5

# Generemos las mediciones
b2 = A2 @ x_fantoma
b4 = A4 @ x_fantoma
b7 = A7 @ x_fantoma

In [7]:
# Ejercicio 6

# Determinamos una semilla aleatoria para garantizar reproducibilidad
SEED = 73
np.random.seed(SEED)

# Creamos el vector z del mismo tamaño que b7, es decir, 119
z = np.random.normal(loc=0.0, scale=1.0, size=119)

# Creamos la lista de desvios (sigma's)
desvios = [0, 0.01, 0.03, 0.06, 0.1]

# Calculamos escala
escala = np.sqrt(np.mean(b7**2))

# Creamos la lista de ruidos (eta's) donde habrá una por cada desvio
ruidos = list()

for ds in desvios:
    ruido = ds * escala * z
    ruidos.append(ruido)

for i in range(len(ruidos)):
    print(ruidos[i])

[ 0.  0.  0.  0. -0. -0.  0. -0. -0.  0.  0. -0.  0. -0.  0. -0.  0. -0.
  0. -0. -0. -0. -0. -0.  0.  0.  0. -0.  0.  0. -0. -0. -0.  0. -0. -0.
  0.  0.  0. -0. -0.  0.  0. -0.  0. -0.  0. -0.  0.  0.  0.  0. -0.  0.
  0. -0.  0.  0.  0. -0.  0. -0. -0.  0. -0.  0.  0.  0. -0.  0.  0. -0.
 -0. -0. -0.  0.  0.  0. -0. -0.  0. -0.  0. -0.  0.  0. -0.  0. -0. -0.
 -0. -0. -0.  0. -0. -0. -0. -0. -0.  0. -0.  0. -0. -0. -0.  0. -0. -0.
  0. -0. -0. -0. -0. -0.  0.  0.  0.  0. -0.]
[ 6.32375691e-03  2.33639200e-02  2.67527858e-02  2.88692346e-03
 -1.64023738e-02 -4.02739078e-04  4.72183396e-03 -1.67680392e-02
 -8.00603360e-03  1.15258522e-02  1.77582235e-02 -1.75961941e-02
  3.62894669e-03 -2.31272753e-03  3.26899152e-03 -1.25647128e-02
  6.30784890e-03 -3.98960650e-03  3.99054175e-04 -5.70708324e-03
 -1.72828631e-04 -5.02571572e-04 -6.77980850e-03 -8.98808417e-03
  5.58916874e-03  1.15180779e-02  1.59002386e-03 -3.73689687e-03
  1.36396788e-02  1.81531824e-02 -2.99293796e-03 -3.94596597e

In [8]:
# Ejercicio 6

# Creamos la lista de mediciones con ruido (b_\sigma)
mediciones_ruidosas = list()

for i in range(len(ruidos)):
    med_ruido = b7 + ruidos[i]
    mediciones_ruidosas.append(med_ruido)

In [9]:
# Ejercicio 6

# Creamos una lista con las reconstrucciones por mínimos cuadrados
reconstrucciones = list()

# Creamos una lista con los residuos cuadrados producto de las reconstrucciones
residuos_cuadrados = list()

for med_ruidosa in mediciones_ruidosas:
    # Resolvemos por mínimos cuadrados
    # Obtenemos el \hat{x}_\sigma
    x_ruidoso = np.linalg.lstsq(A7, med_ruidosa)[0]

    # Actualizamos la lista de reconstrucciones
    reconstrucciones.append(x_ruidoso)

    # Calculamos el residuo cuadrado correspondiente
    res_cuad = np.linalg.norm(A7 @ x_ruidoso - med_ruido)**2

    # Actualizamos la list ade residuos cuadrados
    residuos_cuadrados.append(res_cuad)


Continuaremos utilizando $A_7$ y las mediciones con $\sigma = 0,06$ del ejercicio anterior. La reconstrucción por mínimos cuadrados $\hat{x}_\sigma$ no reproduce exactamente las mediciones ruidosas, sino que genera el
vector $A_7 \hat{x}_\sigma$ más cercano a $b_\sigma$ dentro de $\text{Col}(A_7)$.

1. Calcular el residuo $r_\sigma = b_\sigma − A_7 \hat{x}_\sigma$ y verificar numéricamente que $|| A^{T}_7 r_\sigma || \approx 0$.

In [10]:
# Calculamos el residuo r_sigma
r_sigma = mediciones_ruidosas[3] - A7 @ reconstrucciones[3]

# Verificamos que ||A_7^t * r_sigma|| ~= 0
print(np.linalg.norm(np.array(A7).transpose() @ r_sigma))

4.33947668257523e-14


De este modo, queda verificado númericamente que la norma indicada es apróximadamente cero.

2. Explicar por qué el resultado anterior indica que el residuo es ortogonal a todas las columnas de $A_7$ y que $A_7\hat{x}_\sigma$ es la proyección ortogonal de $b_\sigma$ sobre $\text{Col}(A_7)$.

Por un lado, el resultado obtenido en el inciso previo nos indica que $r_\sigma$ se encuentra aproximadamente en el espacio nulo de A traspuesta. Por teorema, se sabe que $\text{Nul}(A^T) = (\text{Col}A_7)^\perp$. En consecuencia, $r_\sigma$ también es ortogonal al espacio columna de A.

Por otro lado, sabemos que toda solución de mínimos cuadrados (tal como la que obtuvimos previamente) es a su vez la mejor aproximación $\hat{b}$ que se puede obtener respecto a $b_\sigma$. Luego, por teorema de la mejor aproximación, $\hat{b}$ es la proyección ortogonal de $b_\sigma$ sobre el espacio columna de $A_7$.

3. Explicar la diferencia entre las proyecciones tomográficas medidas por el escáner y la proyección
ortogonal que aparece en mínimos cuadrados.

En primer lugar, tenemos la certeza de que las proyecciones tomográficas medidas sin ruido $b$ (contexto ideal) siempre van a derivar en una solución (pues inicialmente la tomografía produjo que $x_{fantoma} \mapsto b$), que además será única a la hora de reconstruir pues $\text{Rango}(A_7) = 64 = n²$ y genera el subespacio donde viven estas mediciciones ideales por lo que $b \in \text{Col}(A_7)$. Sin embargo, al introducir ruido sobre estas $b_\sigma = b + \eta_\sigma$, no tenemos garantía de que $b_\sigma \in\text{Col}(A_7)$ por lo que puede que no haya una reconstrucción exacta (recordemos que $A_7$ no es cuadrada por lo que no hay solución $\forall b\in\mathbb{R}^{119}$).

De esta manera, la proyección ortogonal acerca lo más posible $\hat{b} \in \text{Col}(A_7)$ a $b_\sigma$, que no necesariamente pertenece a dicho subespacio. Luego, esta proyección a diferencia de la otra provee solución para todos los casos más nunca con certeza absoluta (siempre apróximado).